# Notebook 02 — Filter & Extract Exhibition Records

**Project:** Linked Open Exhibition — NFDI4Culture / Hochschule Hannover (BIM-126-02)  
**AI attribution:** GitHub Copilot (Claude Sonnet 4.6)  
**Depends on:** `01_dnb_search.ipynb` — run that first to produce `catalogues/sprengel_raw/`

**Purpose:** Parse the MARC21-XML files from Notebook 01 and extract structured fields for all Sprengel Museum book records into a CSV file.

---

## Background: MARC21

**MARC21 (MAchine-Readable Cataloguing)** is the standard bibliographic metadata format used by libraries worldwide. Records consist of:

- **Control fields** (3-digit tag, no indicators): e.g. `001` (record ID)
- **Data fields** (3-digit tag + 2 indicators + subfields coded with `$a`, `$b`, etc.)

Fields extracted in this notebook:

| Field | Subfield | Meaning |
|---|---|---|
| `001` | — | IDN (unique DNB record identifier) |
| `245` | `a` | Title |
| `264` or `260` | `c` | Publication year |
| `020` | `a` | ISBN |
| `689` | `a` | Subject headings (Schlagwörter) |
| `856` | `u` | URL / link |

In [5]:
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path

RAW_DIR  = Path("../sprengel_raw")
OUT_CSV  = Path("../sprengel_exhibitions.csv")

NS_MARC  = "http://www.loc.gov/MARC21/slim"
NS_SRW   = "http://www.loc.gov/zing/srw/"

xml_files = sorted(RAW_DIR.glob("page_*.xml"))
print(f"Found {len(xml_files)} raw XML file(s) in {RAW_DIR.resolve()}")

Found 6 raw XML file(s) in C:\git\linked-open-exhibition\catalogues\sprengel_raw


## Step 1 — Parse and extract fields

In [6]:
def get_subfield(rec, tag, code):
    """Return first matching subfield value or empty string."""
    field = rec.find(f".//{{{NS_MARC}}}datafield[@tag='{tag}']/{{{NS_MARC}}}subfield[@code='{code}']")
    return (field.text or "").strip() if field is not None else ""

def get_all_subfields(rec, tag, code):
    """Return all matching subfield values as a list."""
    return [
        (el.text or "").strip()
        for el in rec.findall(f".//{{{NS_MARC}}}datafield[@tag='{tag}']/{{{NS_MARC}}}subfield[@code='{code}']")
    ]

rows = []

for xml_file in xml_files:
    root = ET.parse(xml_file).getroot()
    records = root.findall(f".//{{{NS_SRW}}}recordData/{{{NS_MARC}}}record")

    for rec in records:
        idn   = (rec.findtext(f"{{{NS_MARC}}}controlfield[@tag='001']") or "").strip()
        title = get_subfield(rec, "245", "a")
        year  = get_subfield(rec, "264", "c") or get_subfield(rec, "260", "c")
        isbn  = get_subfield(rec, "020", "a")
        url   = get_subfield(rec, "856", "u")
        subjects = " | ".join(get_all_subfields(rec, "689", "a"))

        rows.append({
            "idn":      idn,
            "title":    title,
            "year":     year,
            "isbn":     isbn,
            "url":      url,
            "subjects": subjects,
        })

df_all = pd.DataFrame(rows)
print(f"Total records parsed: {len(df_all)}")
df_all.head(3)

Total records parsed: 582


,idn,title,year,isbn,url,subjects
0,1375818457,Niki de Saint Phalle - Die Grotte,2026,9783775758642,https://deposit.dnb.de/cgi-bin/dokserv?id=b406...,
1,1375817957,Niki de Saint Phalle - The Grotto,2026,9783775758659,https://deposit.dnb.de/cgi-bin/dokserv?id=030d...,
2,1347131019,Feministische Avantgarde,[2025],9783791376226,http://deposit.dnb.de/cgi-bin/dokserv?id=69cec...,Engagierte Kunst | Avantgarde | Künstlerin | ...


## Step 2 — Use all records

All 582 book records are kept. Subject headings are retained in the CSV for reference and later analysis.

In [7]:
df = df_all.copy()
df = df.reset_index(drop=True)

print(f"Total book records: {len(df)}")
df.head()

Total book records: 582


,idn,title,year,isbn,url,subjects
0,1375818457,Niki de Saint Phalle - Die Grotte,2026,9783775758642,https://deposit.dnb.de/cgi-bin/dokserv?id=b406...,
1,1375817957,Niki de Saint Phalle - The Grotto,2026,9783775758659,https://deposit.dnb.de/cgi-bin/dokserv?id=030d...,
2,1347131019,Feministische Avantgarde,[2025],9783791376226,http://deposit.dnb.de/cgi-bin/dokserv?id=69cec...,Engagierte Kunst | Avantgarde | Künstlerin | ...
3,1357144318,Grethe Jürgens,[2025],9783864424571,https://d-nb.info/1357144318/04,"Jürgens, Grethe | Malerei"
4,1361486112,Lillien Grupe - Realität(en)?,[2025],9783949924101,https://d-nb.info/1361486112/04,


## Step 3 — Clean year field

Year values from MARC21 sometimes contain extra punctuation (e.g. `"2023."`). Extract a 4-digit year.

In [8]:
import re

def clean_year(val):
    match = re.search(r"\b(\d{4})\b", str(val))
    return match.group(1) if match else ""

df["year"] = df["year"].apply(clean_year)

print("Year value counts (top 10):")
print(df["year"].value_counts().head(10))

Year value counts (top 10):
year
1997    32
2000    28
1987    26
1996    25
        24
1990    22
1988    21
2021    20
1998    19
2005    18
Name: count, dtype: int64


## Step 4 — Save to CSV

In [9]:
df.to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"Saved {len(df)} records → {OUT_CSV.resolve()}")
df.info()

Saved 582 records → C:\git\linked-open-exhibition\catalogues\sprengel_exhibitions.csv
<class 'pandas.DataFrame'>
RangeIndex: 582 entries, 0 to 581
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   idn       582 non-null    str  
 1   title     582 non-null    str  
 2   year      582 non-null    str  
 3   isbn      582 non-null    str  
 4   url       582 non-null    str  
 5   subjects  582 non-null    str  
dtypes: str(6)
memory usage: 27.4 KB


---

**Next step:** Run `03_dnb_cover_images.ipynb` to retrieve cover images for records that have an ISBN.